# MiniMax H3 — Local Open-Source + Director + LTX on Colab

Optimized for **G4 / RTX PRO 6000 Blackwell 96 GB**.

This notebook now supports:
- **Local open-source H3 Base FL2VA** for text-to-video.
- **Local H3 latent upscale + second-pass refine** used by `H3_Local_OpenSource_MaxQuality_T2V.json`.
- **H3 Ref2VA / Director** workflows.
- **LTX-2.5** in the same ComfyUI runtime.

The local H3 path does **not** use the hosted H3 Max API. It downloads the public H3 weights into Colab.


## 0. Mount Drive + persistence

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PERSIST_MODELS_TO_DRIVE = False   # False = faster model loading from Colab VM disk
PERSIST_OUTPUT_TO_DRIVE = True    # Keep outputs / ComfyUI user data across restarts
DRIVE_ROOT = '/content/drive/MyDrive/MiniMax_H3_ComfyUI'


## 1. Check GPU + safe runtime settings

In [ ]:
import subprocess, os

def sh(cmd):
    return subprocess.check_output(cmd, shell=True, text=True).strip()

gpu_name = sh('nvidia-smi --query-gpu=name --format=csv,noheader | head -n1')
vram_mb = int(sh('nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -n1'))
vram_gb = vram_mb / 1024
name = gpu_name.lower()

if 'rtx pro 6000' in name or 'blackwell' in name:
    H3_RESERVE_VRAM_GB = '6'
elif 'a100' in name:
    H3_RESERVE_VRAM_GB = '4'
elif 'l4' in name:
    H3_RESERVE_VRAM_GB = '2'
else:
    H3_RESERVE_VRAM_GB = '1'

print(f'GPU: {gpu_name} ({vram_gb:.1f} GB)')
print('Reserved VRAM:', H3_RESERVE_VRAM_GB, 'GB')


## 2. Clone the H3 branch

In [ ]:
%cd /content
!rm -rf /content/All-testing /content/minimax_h3_comfy
!git clone --depth 1 --branch minimax-h3-colab https://github.com/Logan17de/All-testing.git /content/All-testing
!cp -r /content/All-testing/video/minimax_h3_comfy /content/minimax_h3_comfy
%cd /content/minimax_h3_comfy


## 3. Install / update ComfyUI + H3 custom nodes

In [ ]:
import os, subprocess, sys
from pathlib import Path

os.environ['COMFY_ROOT'] = '/content/ComfyUI'
os.environ['H3_DRIVE_ROOT'] = DRIVE_ROOT
os.environ['H3_PERSIST_MODELS'] = '1' if PERSIST_MODELS_TO_DRIVE else '0'
os.environ['H3_PERSIST_OUTPUT'] = '1' if PERSIST_OUTPUT_TO_DRIVE else '0'
os.environ['H3_VRAM_MODE'] = 'auto'
os.environ['H3_RESERVE_VRAM_GB'] = H3_RESERVE_VRAM_GB
os.environ['H3_PREVIEW_METHOD'] = 'none'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!bash install_comfy_h3.sh

CUSTOM='/content/ComfyUI/custom_nodes'
def clone_or_pull(url, folder):
    path=f'{CUSTOM}/{folder}'
    if os.path.isdir(path+'/.git'):
        subprocess.run(['git','-C',path,'pull','--ff-only'], check=False)
    else:
        subprocess.run(['git','clone','--depth','1',url,path], check=True)
    req=os.path.join(path,'requirements.txt')
    if os.path.exists(req):
        subprocess.run([sys.executable,'-m','pip','install','-r',req], check=False)

clone_or_pull('https://github.com/AIMixer/ComfyUI_MiniMaxH3_Director.git','ComfyUI_MiniMaxH3_Director')
clone_or_pull('https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git','ComfyUI-VideoHelperSuite')
clone_or_pull('https://github.com/kijai/ComfyUI-KJNodes.git','ComfyUI-KJNodes')
clone_or_pull('https://github.com/pixaroma/ComfyUI-Pixaroma.git','ComfyUI-Pixaroma')

subprocess.run([sys.executable,'-m','pip','install','-U','huggingface_hub'], check=True)
print('Director / VHS / KJNodes / Pixaroma installed or updated.')


## 4. Download H3 models

This cell installs both H3 model families:
- **FL2VA INT8 ConvRot** — local text-to-video / first-last-frame video.
- **Ref2VA INT8 ConvRot** — reference image/video/audio workflows.

It also downloads Qwen3-VL, both H3 VAEs, the Ref2V/FL2V Turbo LoRAs for optional fast workflows, and the H3 3D latent upscaler used by the local max-quality T2V flow.


In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
import os, shutil

MODEL_ROOT = Path(f'{DRIVE_ROOT}/models' if PERSIST_MODELS_TO_DRIVE else '/content/ComfyUI/models')
for folder in ['diffusion_models','text_encoders','vae','loras','latent_upscale_models']:
    (MODEL_ROOT/folder).mkdir(parents=True, exist_ok=True)

# install_comfy_h3.sh already maps the main folders when Drive persistence is enabled.
# Ensure the newer latent-upscale folder is also visible to ComfyUI.
if PERSIST_MODELS_TO_DRIVE:
    local_latent = Path('/content/ComfyUI/models/latent_upscale_models')
    drive_latent = MODEL_ROOT/'latent_upscale_models'
    drive_latent.mkdir(parents=True, exist_ok=True)
    if local_latent.is_symlink() or local_latent.exists():
        if local_latent.is_symlink():
            local_latent.unlink()
        elif local_latent.is_dir():
            shutil.rmtree(local_latent)
        else:
            local_latent.unlink()
    local_latent.symlink_to(drive_latent, target_is_directory=True)

downloads = [
    # Open H3 Base models
    ('Comfy-Org/MiniMax-H3','diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors', MODEL_ROOT, MODEL_ROOT/'diffusion_models'/'minimax_h3_fl2va_pruned_int8_convrot.safetensors'),
    ('Comfy-Org/MiniMax-H3','diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors', MODEL_ROOT, MODEL_ROOT/'diffusion_models'/'minimax_h3_ref2va_pruned_int8_convrot.safetensors'),

    # Shared encoder / VAEs
    ('Comfy-Org/MiniMax-H3','text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', MODEL_ROOT, MODEL_ROOT/'text_encoders'/'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'),
    ('Comfy-Org/MiniMax-H3','vae/minimax_h3_video_vae_fp16.safetensors', MODEL_ROOT, MODEL_ROOT/'vae'/'minimax_h3_video_vae_fp16.safetensors'),
    ('Comfy-Org/MiniMax-H3','vae/minimax_h3_audio_vae_fp32.safetensors', MODEL_ROOT, MODEL_ROOT/'vae'/'minimax_h3_audio_vae_fp32.safetensors'),

    # Optional fast/Turbo workflows
    ('lightx2v/Minimax-h3-Turbo','minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors', MODEL_ROOT/'loras', MODEL_ROOT/'loras'/'minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors'),
    ('lightx2v/Minimax-h3-Turbo','minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors', MODEL_ROOT/'loras', MODEL_ROOT/'loras'/'minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors'),

    # Local second-pass video-latent upscale
    ('LBH-123-AI/Minimax_h3_latent_Upscaler','minimax_h3_latent_upscaler_3d_fp16.safetensors', MODEL_ROOT/'latent_upscale_models', MODEL_ROOT/'latent_upscale_models'/'minimax_h3_latent_upscaler_3d_fp16.safetensors'),
]

for repo_id, filename, local_dir, target in downloads:
    if target.exists() and target.stat().st_size > 1024*1024:
        print('SKIP', target.name)
        continue
    print('DOWNLOAD', filename)
    hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(local_dir))
    if not target.exists():
        raise FileNotFoundError(f'Expected model was not created: {target}')

print('H3 model stack ready at:', MODEL_ROOT)


## 5. Restore / upload your H3 workflow JSONs

Workflow JSONs are persisted separately in Drive so a fresh Colab runtime can restore them automatically.

For the new local T2V pipeline, use:
`H3_Local_OpenSource_MaxQuality_T2V.json`

For the fixed Director partial-render flow, use:
`Logan_H3_Director_SelectedClips_Corrected.json`


In [ ]:
from pathlib import Path
import shutil

DRIVE_WORKFLOWS = Path(DRIVE_ROOT) / 'workflows'
COMFY_WORKFLOWS = Path('/content/ComfyUI/user/default/workflows')
DRIVE_WORKFLOWS.mkdir(parents=True, exist_ok=True)
COMFY_WORKFLOWS.mkdir(parents=True, exist_ok=True)

restored = []
for src in DRIVE_WORKFLOWS.glob('*.json'):
    dst = COMFY_WORKFLOWS / src.name
    shutil.copy2(src, dst)
    restored.append(dst.name)

print('Restored workflows:', restored if restored else 'none yet')
print('Drive workflow folder:', DRIVE_WORKFLOWS)


### Optional: upload a workflow JSON now

In [ ]:
UPLOAD_WORKFLOW_NOW = False

if UPLOAD_WORKFLOW_NOW:
    from google.colab import files
    uploaded = files.upload()
    for name, blob in uploaded.items():
        if not name.lower().endswith('.json'):
            print('Skipping non-JSON:', name)
            continue
        drive_dst = DRIVE_WORKFLOWS / name
        comfy_dst = COMFY_WORKFLOWS / name
        drive_dst.write_bytes(blob)
        comfy_dst.write_bytes(blob)
        print('Installed + persisted:', name)
else:
    print('Set UPLOAD_WORKFLOW_NOW = True and rerun this cell when you want to upload a workflow JSON.')


## 6. Install LTX-2.5 models + flows (optional)

Uses the same `/content/ComfyUI`. LTX-2.5 is separate from H3; keep this cell if you want the B-roll / image-animation workflows too.


In [ ]:
import os, subprocess, sys
from google.colab import userdata

try:
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
except Exception:
    pass

INSTALL_LTX25 = True
if INSTALL_LTX25:
    LTX_MODEL_ROOT = f'{DRIVE_ROOT}/models' if PERSIST_MODELS_TO_DRIVE else '/content/ComfyUI/models'
    cmd = [
        sys.executable,
        '/content/minimax_h3_comfy/download_ltx_models.py',
        '--comfy-root', '/content/ComfyUI',
        '--model-root', LTX_MODEL_ROOT,
    ]
    subprocess.run(cmd, check=True)
    print('LTX-2.5 ready.')
else:
    print('LTX-2.5 skipped.')


## 7. Launch ComfyUI

In [ ]:
!bash launch_comfy.sh


## First run

### Local open-source H3 text-to-video
Open **`H3_Local_OpenSource_MaxQuality_T2V.json`** after copying/uploading it in section 5.

The workflow uses:
1. Open H3 Base FL2VA at native 1344×768.
2. Native H3 video + audio generation.
3. H3 3D video-latent upscale.
4. A second low-denoise H3 sampling pass for the local max-quality output.

The second pass is a local replacement for the unavailable hosted `H3-Regenerate-2K`; it is not the closed API model.

### Director / reference generation
Open your corrected Director workflow JSON. In **segments** export mode the Director saves full MP4s directly under:

`MyDrive/MiniMax_H3_ComfyUI/output/minimax_seg_export/<timestamp>/`

Do not attach a VHS Video Combine to a one-frame released poster and expect it to be the full clip.

### LTX-2.5
Use `LTX_2.5_T2V.json`, `LTX_2.5_I2V.json`, or `LTX_2.5_FLF2V.json` when needed.
